In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import torch
from torch.utils.data import DataLoader

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from src.trainer import SimpleTrainer, SIBufferTrainer
from src.data_utils import (
    get_mnist_tasks,
    _extract_targets,
    get_context_sets,
    create_holdout_set,
)
from src.utils import InContextHead
from src import models
from src.buffer import MultiTaskBuffer

### Class Incremental Learning

In [4]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

Tasks: [[3, 4], [8, 9], [6, 7], [0, 5], [1, 2]]


In [5]:
train_tasks, buffer_sets = zip(
    *[create_holdout_set(dataset) for dataset in train_tasks]
)
print([len(task) for task in buffer_sets])
print([len(task) for task in train_tasks])

[956, 945, 973, 910, 1014]
[8611, 8505, 8762, 8193, 9131]


In [ ]:
model = models.get_mnist_model(device="cuda")

trainer = SimpleTrainer(model, seed=42)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:2])

Training Epochs: 100%|█████████████████████████████████████████| 3/3 [00:35<00:00, 11.85s/it, val_loss=0.0371, val_acc=0.99]


Test Results: [(0.0218, 0.9925), (31.9972, 0.0)]


[(0.0218, 0.9925), (31.9972, 0.0)]

In [ ]:
buffer = MultiTaskBuffer([])
interval_trainer = SIBufferTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=200,
    min_acc_limit=0.8,
    primal_learning_rate=0.5,
    paradigm="CIL",
    buffer=buffer,
    seed=42,
)

loader = DataLoader(
    train_tasks[1],
    batch_size=128,
    shuffle=True,
    generator=torch.Generator().manual_seed(42),
)
t1_sample = next(iter(loader))
# Compute bounds for task 0
interval_trainer.compute_rashomon_set(test_tasks[0], prune_prop=0.8, si_batch=t1_sample)

interval_trainer.add_to_buffer(buffer_sets[0])

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    interval_trainer.train(train, val, batch_size=256, target_acc=0.9)
    interval_trainer.test(test_tasks[0 : i + 1])
    if i < len(train_tasks) - 2:
        loader = DataLoader(
            train_tasks[i + 1],
            batch_size=128,
            shuffle=True,
            generator=torch.Generator().manual_seed(42),
        )
        sample = next(iter(loader))
        interval_trainer.compute_rashomon_set(test, prune_prop=0.8, si_batch=sample)
        interval_trainer.add_to_buffer(buffer_sets[i])

To keep top 20%, found global SI threshold: 0.0062
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.1883 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.80
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.99,  Min acc soft=0.99


100%|█████████████████████████████████████████| 200/200 [00:13<00:00, 14.33it/s, size=174.30, obj=0.028, min_soft_acc=0.828]


Final bbox:  Obj=0.03,  Size=174.30,  Min acc hard=0.79,  Min acc soft=0.79
Computing final certificates over 256 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['5.51', '23.81', '57.01', '77.27', '94.76', '111.25', '127.84', '140.81', '158.96', '174.30']
Checkpoint certificates: ['0.93', '0.98', '0.89', '0.86', '0.86', '0.84', '0.80', '0.86', '0.79', '0.79']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs:   1%|▎                            | 1/100 [00:10<16:43, 10.14s/it, val_loss=5.6024, val_acc=0.0055, proj=2]


Early stopping at epoch 2
Consume buffer data.
To keep top 20%, found global SI threshold: 0.0019
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.1800 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.80
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.98,  Min acc soft=0.98


...

Early stopping at epoch 1
Test Results: [(0.3075, 0.8966), (0.9993, 0.6324), (15.53, 0.0)]
To keep top 20%, found global SI threshold: 0.0050
---------------------------- Computing Rashomon set ----------------------------


Training Epochs:   0%|                                    | 0/100 [00:03<?, ?it/s, val_loss=26.6402, val_acc=0.0000, proj=0]


Early stopping at epoch 1
Test Results: [(0.3075, 0.8966), (0.9993, 0.6324), (21.3275, 0.0), (26.7711, 0.0)]


Training Epochs:   0%|                                    | 0/100 [00:04<?, ?it/s, val_loss=20.5003, val_acc=0.0000, proj=0]


Early stopping at epoch 1
Test Results: [(0.3075, 0.8966), (0.9993, 0.6324), (27.6672, 0.0), (26.7711, 0.0), (20.9222, 0.0)]


### Task Incremental Learning

In [ ]:
context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cuda")
head.set_context(0)
model = models.get_mnist_model(device="cuda", head=head)

trainer = SimpleTrainer(model)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:2])

Training Epochs: 100%|█████████████████████████████████████████| 3/3 [00:16<00:00,  5.40s/it, val_loss=0.0407, val_acc=0.99]


Test Results: [(0.0227, 0.9925), (22.4285, 0.0)]


[(0.0227, 0.9925), (22.4285, 0.0)]

In [ ]:
buffer = MultiTaskBuffer([])
interval_trainer = SIBufferTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=200,
    min_acc_limit=0.8,
    primal_learning_rate=0.5,
    paradigm="TIL",
    buffer=buffer,
    seed=42,
)

loader = DataLoader(
    train_tasks[1],
    batch_size=128,
    shuffle=True,
    generator=torch.Generator().manual_seed(42),
)
t1_sample = next(iter(loader))
# Compute bounds for task 0

interval_trainer.compute_rashomon_set(
    test_tasks[0], prune_prop=0.8, si_batch=t1_sample, context_id=0, si_context_id=1
)

interval_trainer.add_to_buffer(buffer_sets[0])

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    interval_trainer.train(train, val, batch_size=256, target_acc=0.9, context_id=i)
    interval_trainer.test(test_tasks[0 : i + 1], context_list=list(range(0, i + 1)))
    if i < len(train_tasks) - 2:
        loader = DataLoader(
            train_tasks[i + 1],
            batch_size=128,
            shuffle=True,
            generator=torch.Generator().manual_seed(42),
        )
        sample = next(iter(loader))
        interval_trainer.compute_rashomon_set(
            test, prune_prop=0.8, si_batch=sample, context_id=i, si_context_id=i + 1
        )
        interval_trainer.add_to_buffer(buffer_sets[i])

To keep top 20%, found global SI threshold: 0.0003
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.1883 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.80
Initial bbox:  Obj=14.94,  Size=92320.00,  Min acc hard=0.99,  Min acc soft=0.99


100%|██████████████████████████████████████| 200/200 [00:09<00:00, 20.20it/s, size=92320.99, obj=14.941, min_soft_acc=0.952]


Final bbox:  Obj=14.94,  Size=92320.99,  Min acc hard=0.95,  Min acc soft=0.95
Computing final certificates over 256 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['92321.08', '92320.12', '92321.27', '92320.14', '92320.34', '92321.32', '92321.57', '92320.22', '92320.26', '92320.99']
Checkpoint certificates: ['0.88', '0.99', '0.91', '0.99', '0.99', '0.89', '0.85', '0.99', '0.99', '0.95']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs:   2%|▌                            | 2/100 [00:12<10:23,  6.36s/it, val_loss=0.1541, val_acc=0.9579, proj=6]


Early stopping at epoch 3
Test Results: [(0.0212, 0.9925), (0.1397, 0.9632)]
To keep top 20%, found global SI threshold: 0.0002
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.1549 (Positive = violated)
Computing Rashomon set within outer box of size: 92321.57
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.80
Initial bbox:  Obj=14.94,  Size=92320.00,  Min acc hard=0.96,  Min acc soft=0.95


...

Final bbox:  Obj=7.47,  Size=46160.42,  Min acc hard=0.98,  Min acc soft=0.98
Computing final certificates over 256 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['46160.42', '46160.42', '46160.42', '46160.42', '46160.42', '46160.42', '46160.42', '46160.42', '46160.42', '46160.42']
Checkpoint certificates: ['0.98', '0.98', '0.98', '0.98', '0.98', '0.98', '0.98', '0.98', '0.98', '0.98']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs:   2%|▌                            | 2/100 [00:11<09:24,  5.76s/it, val_loss=0.1230, val_acc=0.9585, proj=0]


Early stopping at epoch 3
Test Results: [(0.0212, 0.9925), (0.1398, 0.9632), (0.0211, 0.9935), (0.0983, 0.9696)]


Training Epochs:   6%|█▋                           | 6/100 [00:38<09:56,  6.34s/it, val_loss=0.0552, val_acc=0.9832, proj=0]


Early stopping at epoch 7
Test Results: [(0.0212, 0.9925), (0.1398, 0.9632), (0.0211, 0.9935), (0.0983, 0.9696), (0.0631, 0.9889)]


### Domain Incremental Learning

In [12]:
def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
    """Map the global label to the in context label."""
    return labels % 2

In [ ]:
model = models.get_mnist_model(device="cuda")

trainer = SimpleTrainer(model, paradigm="DIL", domain_map_fn=domain_map_fn)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:2])

Training Epochs: 100%|█████████████████████████████████████████| 3/3 [00:16<00:00,  5.41s/it, val_loss=0.038, val_acc=0.989]


Test Results: [(0.0216, 0.993), (3.5566, 0.1478)]


[(0.0216, 0.993), (3.5566, 0.1478)]

In [ ]:
buffer = MultiTaskBuffer([])
interval_trainer = SIBufferTrainer(
    trainer.model,
    checkpoint=20,
    n_iters=200,
    min_acc_limit=0.8,
    primal_learning_rate=0.5,
    paradigm="DIL",
    buffer=buffer,
    domain_map_fn=domain_map_fn,
    seed=42,
)

loader = DataLoader(
    train_tasks[1],
    batch_size=128,
    shuffle=True,
    generator=torch.Generator().manual_seed(42),
)
t1_sample = next(iter(loader))
# Compute bounds for task 0

interval_trainer.compute_rashomon_set(test_tasks[0], prune_prop=0.8, si_batch=t1_sample)

interval_trainer.add_to_buffer(buffer_sets[0])

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    interval_trainer.train(train, val, batch_size=256, target_acc=0.9)
    interval_trainer.test(test_tasks[0 : i + 1])
    if i < len(train_tasks) - 2:
        loader = DataLoader(
            train_tasks[i + 1],
            batch_size=128,
            shuffle=True,
            generator=torch.Generator().manual_seed(42),
        )
        sample = next(iter(loader))
        interval_trainer.compute_rashomon_set(test, prune_prop=0.8, si_batch=sample)
        interval_trainer.add_to_buffer(buffer_sets[i])

To keep top 20%, found global SI threshold: 0.0003
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.1886 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.80
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=0.99,  Min acc soft=0.99


100%|██████████████████████████████████████████| 200/200 [00:09<00:00, 20.33it/s, size=48.61, obj=0.008, min_soft_acc=0.816]


Final bbox:  Obj=0.01,  Size=48.61,  Min acc hard=0.82,  Min acc soft=0.81
Computing final certificates over 256 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['4.93', '18.61', '28.82', '35.02', '37.54', '41.76', '43.74', '46.24', '46.28', '48.61']
Checkpoint certificates: ['0.92', '0.85', '0.80', '0.80', '0.84', '0.80', '0.81', '0.81', '0.82', '0.82']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs:   0%|                                     | 0/100 [00:01<?, ?it/s, val_loss=3.0085, val_acc=0.1774, proj=0]


Early stopping at epoch 1
Consume buffer data.
To keep top 20%, found global SI threshold: 0.0003
---------------------------- Computing Rashomon set ----------------------------
Initial acc constraint violation: -0.1950 (Positive = violated)
Number of model parameters: 6179
Computing Rashomon set with min acc limit: 0.80
Initial bbox:  Obj=0.00,  Size=0.00,  Min acc hard=1.00,  Min acc soft=0.99


...

Final bbox:  Obj=0.00,  Size=2.59,  Min acc hard=0.82,  Min acc soft=0.82
Computing final certificates over 256 samples
Checkpointed every 20 iterations for a total of 10 checkpoints
Checkpoints sizes: ['1.76', '2.50', '2.90', '2.89', '2.32', '2.73', '2.85', '2.60', '3.01', '2.59']
Checkpoint certificates: ['0.81', '0.83', '0.80', '0.80', '0.83', '0.82', '0.81', '0.82', '0.78', '0.82']
----------------------- Finished Computing Rashomon set ------------------------


Training Epochs:   0%|                                     | 0/100 [00:02<?, ?it/s, val_loss=2.7143, val_acc=0.3927, proj=0]


Early stopping at epoch 1
Test Results: [(0.0345, 0.992), (2.1553, 0.2864), (0.2998, 0.8701), (2.531, 0.4092)]


Training Epochs:   0%|                                     | 0/100 [00:01<?, ?it/s, val_loss=1.6372, val_acc=0.5452, proj=0]


Early stopping at epoch 1
Test Results: [(0.0348, 0.992), (2.1618, 0.2879), (0.301, 0.8701), (2.5392, 0.4081), (1.9348, 0.5127)]
